# Semantic Graph Pipeline — Stage 1: Code Extraction

This notebook orchestrates `extract.py` — the first stage of the knowledge graph pipeline.

It discovers services in a monorepo, parses source code with tree-sitter, and extracts:
- HTTP endpoints (routes)
- Inter-service calls
- Dependencies
- Annotations and framework markers


## Configuration

Set your parameters here before running extraction.

In [1]:
from pathlib import Path
import json
import subprocess

# Configuration
repo_root = "./repos/microservices-demo"  # Path to your monorepo or cloned services
output_file = "intermediate.json"   # Output file path
skip_cdxgen = False                  # Set to False once cdxgen is installed
service_filter = None               # Set to a service name to debug a single service (e.g., "cartservice")
verbose = False                     # Enable detailed logging

print("Configuration:")
print(f"  repo_root: {repo_root}")
print(f"  output_file: {output_file}")
print(f"  skip_cdxgen: {skip_cdxgen}")
print(f"  service_filter: {service_filter}")
print(f"  verbose: {verbose}")

Configuration:
  repo_root: ./repos/microservices-demo
  output_file: intermediate.json
  skip_cdxgen: False
  service_filter: None
  verbose: False


## Run Extraction

Execute extract.py with the configured parameters.

In [2]:
# Build command with uv run to ensure correct environment
cmd = [
    "uv", "run", "python",
    "extract.py",
    "--repo-root", repo_root,
    "--output", output_file,
]

if skip_cdxgen:
    cmd.append("--skip-cdxgen")

if service_filter:
    cmd.extend(["--service", service_filter])

if verbose:
    cmd.append("--verbose")

print("Running:", " ".join(cmd))
print()

# Run extract.py and capture output
result = subprocess.run(cmd, capture_output=True, text=True)

# Always show stdout and stderr
if result.stdout:
    print(result.stdout)
if result.stderr:
    print("STDERR:")
    print(result.stderr)

if result.returncode == 0:
    print("✓ Extraction completed successfully")
else:
    print(f"✗ Extraction failed with exit code {result.returncode}")

Running: uv run python extract.py --repo-root ./repos/microservices-demo --output intermediate.json

STDERR:
Uninstalled 1 package in 5ms
Installed 1 package in 7ms
14:42:14  INFO      Initialised tree-sitter parsers for: java, python, javascript, typescript, go
14:42:14  INFO      Discovered 11 service(s) under C:\Users\EwegConsulting\Documents\Projects\semantic_graph\repos\microservices-demo
14:42:14  INFO      Services to process: ['adservice', 'checkoutservice', 'currencyservice', 'emailservice', 'frontend', 'loadgenerator', 'paymentservice', 'productcatalogservice', 'recommendationservice', 'shippingservice', 'shoppingassistantservice']
14:42:14  INFO      Processing: adservice
14:42:14  INFO        Language: java  |  Source files: 2
14:42:14  INFO        gRPC services (from .proto): ['CartService', 'RecommendationService', 'ProductCatalogService', 'ShippingService', 'CurrencyService', 'PaymentService', 'EmailService', 'CheckoutService', 'AdService']
14:42:14  INFO        Running 

## Results Summary

Load and summarize the extracted data.

In [3]:
# Load results
output_path = Path(output_file)

if output_path.exists():
    with open(output_path) as f:
        data = json.load(f)
    
    meta = data["meta"]
    services = data["services"]
    
    print("Extraction Metadata:")
    print(f"  Repo: {meta['repo_root']}")
    print(f"  Time: {meta['extracted_at']}")
    print(f"  cdxgen_skipped: {meta['cdxgen_skipped']}")
else:
    print(f"Output file not found: {output_file}")
    services = []

Extraction Metadata:
  Repo: C:\Users\EwegConsulting\Documents\Projects\semantic_graph\repos\microservices-demo
  Time: 2026-04-15T12:42:32.198515+00:00
  cdxgen_skipped: False


In [4]:
# Calculate summary statistics
total_deps = sum(len(s["dependencies"]) for s in services)
total_endpoints = sum(len(s["endpoints"]) for s in services)
total_calls = sum(len(s["calls"]) for s in services)

print("\n" + "="*60)
print("EXTRACTION SUMMARY")
print("="*60)
print(f"Services extracted:     {len(services)}")
print(f"Total dependencies:     {total_deps}")
print(f"Total endpoints:        {total_endpoints}")
print(f"Total service calls:    {total_calls}")
print("="*60)


EXTRACTION SUMMARY
Services extracted:     11
Total dependencies:     1104
Total endpoints:        60
Total service calls:    12


In [ ]:
# Services breakdown table
header = f"{'Service':<30} {'Lang':<12} {'Framework':<25} {'Endpoints':>9} {'Calls':>6} {'Deps':>6} {'Classes':>8}"
print(header)
print("-" * len(header))
for svc in sorted(services, key=lambda s: s["name"]):
    print(
        f"{svc['name']:<30} "
        f"{svc.get('language', '—'):<12} "
        f"{svc.get('framework', '—'):<25} "
        f"{len(svc['endpoints']):>9} "
        f"{len(svc['calls']):>6} "
        f"{len(svc['dependencies']):>6} "
        f"{len(svc.get('classes', [])):>8}"
    )

## Service Details

Inspect extraction results for a specific service.

In [12]:
[s["name"] for s in services]

['adservice',
 'checkoutservice',
 'currencyservice',
 'emailservice',
 'frontend',
 'loadgenerator',
 'paymentservice',
 'productcatalogservice',
 'recommendationservice',
 'shippingservice',
 'shoppingassistantservice']

In [13]:
# Select a service to inspect
if services:
    service_name = services[2]["name"]  # Change this to inspect a different service
    service = next((s for s in services if s["name"] == service_name), None)
    
    if service:
        print(f"\nService: {service['name']}")
        print(f"Framework: {service.get('framework', '—')}")
        print(f"Path: {service.get('path', '—')}")
        
        print(f"\n📍 Endpoints ({len(service['endpoints'])}):\n")
        for ep in service["endpoints"][:10]:
            protocol = ep.get('protocol', '?')
            print(f"  {protocol:6} {ep.get('path', 'unknown')}")
        if len(service["endpoints"]) > 10:
            print(f"  ... and {len(service['endpoints']) - 10} more")
        
        print(f"\n→ Service Calls ({len(service['calls'])}):\n")
        # calls is a list of service names (strings)
        if service["calls"]:
            for target in service["calls"]:
                print(f"  • {target}")
        else:
            print("  (none)")
        
        print(f"\n📦 Dependencies ({len(service['dependencies'])}):\n")
        for dep in service["dependencies"][:10]:
            print(f"  {dep.get('name', 'unknown')} v{dep.get('version', '?')}")
        if len(service["dependencies"]) > 10:
            print(f"  ... and {len(service['dependencies']) - 10} more")


Service: currencyservice
Framework: gRPC (grpc-js)
Path: C:\Users\EwegConsulting\Documents\Projects\semantic_graph\repos\microservices-demo\src\currencyservice

📍 Endpoints (16):

  grpc   AddItem
  grpc   GetCart
  grpc   EmptyCart
  grpc   ListRecommendations
  grpc   ListProducts
  grpc   GetProduct
  grpc   SearchProducts
  grpc   GetQuote
  grpc   ShipOrder
  grpc   GetSupportedCurrencies
  ... and 6 more

→ Service Calls (0):

  (none)

📦 Dependencies (272):

  profiler v6.0.4
  common v5.0.2
  projectify v4.0.0
  promisify v4.0.0
  arrify v2.0.1
  duplexify v4.1.2
  end-of-stream v1.4.4
  once v1.4.0
  wrappy v1.0.2
  inherits v2.0.4
  ... and 262 more


In [51]:
from pathlib import Path

service_path = Path("./repos/microservices-demo/src/checkoutservice")
if service_path.exists():
    for item in sorted(service_path.rglob("*"))[:20]:  # First 20 items
        print(item.relative_to(service_path))
else:
    print(f"Path not found: {service_path}")

.dockerignore
Dockerfile
genproto
genproto\demo.pb.go
genproto\demo_grpc.pb.go
genproto.sh
go.mod
go.sum
main.go
money
money\money.go
money\money_test.go
README.md


## Stage 2: Neo4j Graph Construction

Load the extracted service data into Neo4j to build a queryable knowledge graph.

The graph contains:
- **Service** nodes: microservices with language, framework, and git metadata
- **Endpoint** nodes: HTTP routes and gRPC methods, linked to services
- **Library** nodes: dependencies (deduplicated by PURL), with version/scope info
- **GrpcService** nodes: gRPC service contracts from .proto files

Relationships represent:
- **CALLS**: service-to-service invocations (12 edges)
- **EXPOSES**: service-to-endpoint relationships  (60 edges)
- **DEPENDS_ON**: service-to-library dependencies (1005+ edges)
- **IMPLEMENTS**: service-to-gRPC-service contracts
- **USES_GRPC**: service-to-gRPC-client relationships

Use `graph.py` to load the graph, then query it in Neo4j Browser at http://localhost:7474

In [52]:
import subprocess

# Neo4j connection config
neo4j_uri      = "bolt://localhost:7687"
neo4j_user     = "neo4j"
neo4j_password = "password"
clear_first    = True   # Set False for idempotent MERGE; True to wipe and rebuild

# Before running this cell, start Neo4j with:
# docker run --name neo4j -p 7474:7474 -p 7687:7687 \
#   -e NEO4J_AUTH=neo4j/password --restart unless-stopped -d neo4j:latest

cmd = [
    "uv", "run", "python",
    "graph.py",
    "--input",          "intermediate.json",
    "--neo4j-uri",      neo4j_uri,
    "--neo4j-user",     neo4j_user,
    "--neo4j-password", neo4j_password,
]
if clear_first:
    cmd.append("--clear")

print("Running:", " ".join(cmd))
print()

result = subprocess.run(cmd, capture_output=True, text=True)

if result.stdout:
    print(result.stdout)
if result.stderr:
    print("STDERR:")
    print(result.stderr)

if result.returncode == 0:
    print("\n✓ Graph construction completed successfully")
    print(f"Open Neo4j Browser: http://localhost:7474")
    print(f"Credentials: {neo4j_user} / {neo4j_password}")
else:
    print(f"\n✗ Graph construction failed with exit code {result.returncode}")

Running: uv run python graph.py --input intermediate.json --neo4j-uri bolt://localhost:7687 --neo4j-user neo4j --neo4j-password password --clear


GRAPH SUMMARY
Nodes:
  Service             : 11
  Endpoint            : 60
  Library             : 509
  GrpcService         : 10
  Class               : 31
  Function            : 42

Relationships:
  CALLS               : 4
  EXPOSES             : 60
  DEPENDS_ON          : 1013
  IMPLEMENTS          : 32
  USES_GRPC           : 17
  HAS_CLASS           : 31
  HAS_METHOD          : 42
  EXTENDS             : 0

Service Call Graph:
  frontend -> adservice
  frontend -> checkoutservice
  frontend -> currencyservice
  recommendationservice -> productcatalogservice

Top 10 Most-used Libraries:
  4 services  opentelemetry-api @ 1.39.1
  4 services  charset-normalizer @ 3.3.2
  4 services  certifi @ 2024.7.4
  4 services  python-dateutil @ 2.9.0.post0
  4 services  google-auth @ 2.23.4
  4 services  opentelemetry-instrumentation-grpc @ 0.60b1
  

## Stage 3: Graph Verification

Connect directly to Neo4j and verify the graph was built correctly.

Query the service call graph, explore the most-used libraries, and check node counts by type.

In [ ]:
from neo4j import GraphDatabase

# Use same config as Stage 2
driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))

try:
    with driver.session() as session:
        # Node counts
        counts = {}
        for label in ["Service", "Endpoint", "Library", "GrpcService"]:
            result = session.run(f"MATCH (n:{label}) RETURN count(n) AS c")
            counts[label] = result.single()["c"]
        
        # Relationship counts
        rel_counts = {}
        for rel_type in ["CALLS", "EXPOSES", "DEPENDS_ON", "IMPLEMENTS", "USES_GRPC"]:
            result = session.run(f"MATCH ()-[r:{rel_type}]->() RETURN count(r) AS c")
            rel_counts[rel_type] = result.single()["c"]
        
        # Service call graph
        calls = session.run(
            "MATCH (a:Service)-[:CALLS]->(b:Service) RETURN a.name AS caller, b.name AS callee ORDER BY caller"
        ).data()
        
        # Most-depended-on libraries
        top_libs = session.run("""
            MATCH (s:Service)-[:DEPENDS_ON]->(l:Library)
            RETURN l.name AS lib, l.version AS version, count(s) AS services
            ORDER BY services DESC LIMIT 10
        """).data()

finally:
    driver.close()

# Display results
print("=" * 60)
print("GRAPH VERIFICATION")
print("=" * 60)

print("\nNode Counts:")
for label, count in counts.items():
    print(f"  {label:20s}: {count}")

print("\nRelationship Counts:")
for rel_type, count in rel_counts.items():
    print(f"  {rel_type:20s}: {count}")

print("\nService Call Graph (12 edges):")
for row in calls:
    print(f"  {row['caller']:30s} → {row['callee']}")

print("\nTop 10 Most-used Libraries:")
for row in top_libs:
    print(f"  {row['services']} services  {row['lib']:40s} @ {row['version']}")

print("=" * 60)
print(f"\n✓ Graph verification complete. Nodes: {sum(counts.values())}, Edges: {sum(rel_counts.values())}")